# 🌳 Where Are the Trees? — Mapping Vegetation (NDVI) Across CDMX

**Measuring green from space to find the second half of the urban-heat story.**

*Notebook 2 of 8 · Project: Dos Méxicos Bajo el Mismo Sol*  
*Author: Nelly Itzel Rodríguez Ortiz · Last updated: 4 June 2026*

Notebook 01 showed us that northern CDMX is up to **8 °C hotter** than the south. The first question anyone asks: *is it because there are fewer trees?* Today we answer that — by measuring green from space.

## 🔁 Quick recap from Notebook 01

In the previous notebook we used Landsat 8 and 9 to measure **Land Surface Temperature (LST)** across the ZMVM — the 16 alcaldías of CDMX plus the 60 EdoMex municipios that make up the metropolitan area. We found a **5–8 °C gap** between the northeast and the south, in both summer and winter.

> The southern mountains (Desierto de los Leones, the Ajusco, the Pedregal) stay cool and green. The urban north glows red.

## ❓ The first suspect: vegetation

That 5–8 °C gap is large enough to shape daily life. So what causes it? The first suspect is the most intuitive one: **vegetation**.

Trees and parks cool a city in two ways: they cast **shade**, and they release water from their leaves into the air, a process called **evapotranspiration**. Less vegetation means less of both, and therefore more heat.

In this notebook we measure that cooling capacity from space, using a satellite index called **NDVI**.

## 🌿 What is NDVI?

**NDVI = Normalized Difference Vegetation Index**

It is a single number, between **−1 and +1**, that measures "greenness" from space. The idea is simple: healthy plants reflect a lot of **near-infrared** light (NIR) and absorb **red** light to power photosynthesis. The ratio of those two signals is a near-perfect proxy for how much chlorophyll is out there.

| Surface | Typical NDVI |
|---|---|
| Dense forest, healthy woodland (Ajusco, Desierto de los Leones) | **0.6 – 0.9** |
| Urban parks, irrigated cropland (Chapultepec, Coyoacán plazas) | **0.4 – 0.6** |
| Residential streets with some trees | **0.2 – 0.4** |
| Dense urban, concrete, asphalt (Iztapalapa, GAM) | **0.05 – 0.2** |
| Water (lakes, reservoirs) | **less than 0** |

> 🌱 *If LST is the city's thermometer, NDVI is its chlorophyll detector.*

### 🎨 A note on the colors

We chose the palette on purpose. **White means concrete** — the streets, roofs, and pavement that absorb and re-radiate heat. The greens go from pale (sparse grass) to deep, dark green (dense forest). The white at the bottom of the scale is not a default — it is a deliberate choice to evoke the asphalt that dominates the urban north, and to connect visually with the LST story of Notebook 01.

## 🗂️ Where the data comes from

| Step | Detail |
|---|---|
| **Satellite** | Landsat 8 and Landsat 9 (NASA–USGS) |
| **Product** | Collection 2, Level 2 (surface reflectance) |
| **Bands used** | SR_B4 (Red) and SR_B5 (Near-Infrared) |
| **Formula** | `NDVI = (NIR − Red) / (NIR + Red)` |
| **Time windows** | Summer 2025 (Jun–Aug) and Winter 2025–2026 (Dec–Feb) |
| **Cloud mask** | QA_PIXEL bits 3 (cloud) and 4 (shadow) |

All the heavy lifting (cloud masking, NDVI calculation, median compositing) is done by the `src/` package. This notebook focuses on **telling the story**.

### ⚙️ One-time setup

The first time you run this notebook, uncomment the `ee.Authenticate()` line and follow the browser prompt. After that, the kernel remembers you for the session.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `import src` works from any CWD

import ee
import geemap

from src import (
    CDMX_CENTER, CDMX_ZOOM, EE_PROJECT_ID, NDVI_VIS_PARAMS,
    get_zmvm_municipalities_aoi,
    load_ndvi_composite,
)

# ee.Authenticate()  # ← uncomment the first time
ee.Initialize(project=EE_PROJECT_ID)
print("✅ Earth Engine ready")

## 🔧 Building the map, step by step

The pipeline mirrors Notebook 01, but instead of temperature we measure "greenness":

1. **Collect** every clear-sky Landsat 8/9 image of the ZMVM for the season.
2. **Mask out the clouds** using the QA_PIXEL bitmask (bit 3 = cloud, bit 4 = shadow).
3. **Calculate NDVI** from the Red (SR_B4) and Near-Infrared (SR_B5) bands.
4. **Take the median** of all the clean images, to ignore stray readings the way the **house price in the middle of the street** ignores the one mansion that would skew an average.

The result: a single, clean vegetation map for the whole metropolitan area.

In [ ]:
aoi = get_zmvm_municipalities_aoi()   # ZMVM: 16 CDMX alcaldías + 60 EdoMex municipios
print("AOI features:", aoi.size().getInfo())

In [ ]:
ndvi_summer = load_ndvi_composite(
    aoi, start_date="2025-06-01", end_date="2025-09-01", cloud_max=20.0,
)
ndvi_winter = load_ndvi_composite(
    aoi, start_date="2025-12-01", end_date="2026-03-01", cloud_max=20.0,
)

print("Summer composite — band:", ndvi_summer.bandNames().getInfo())
print("Winter composite — band:", ndvi_winter.bandNames().getInfo())

## 🗺️ Map: Mexico City's vegetation, summer vs winter

Use the **layer control** (top right) to toggle between seasons. Hover over the map to read the NDVI value at any point. **White is concrete; deep green is forest.**

In [ ]:
Map = geemap.Map(center=CDMX_CENTER, zoom=CDMX_ZOOM)

# Both seasons as toggleable raster layers.
Map.addLayer(ndvi_summer, NDVI_VIS_PARAMS, "🌿 NDVI — Summer 2025 (Jun–Aug)")
Map.addLayer(ndvi_winter, NDVI_VIS_PARAMS, "🌲 NDVI — Winter 2025–2026 (Dec–Feb)")

# ZMVM boundary for context.
Map.addLayer(
    ee.Image().paint(aoi, 0, 2), {"palette": ["white"]}, "ZMVM boundary",
)

# Colorbar keyed to the summer layer.
Map.add_colorbar(
    NDVI_VIS_PARAMS,
    label="NDVI (greenness, 0.1–0.8)",
    layer_name="🌿 NDVI — Summer 2025 (Jun–Aug)",
)
Map.add_layer_control()
Map

> 🔍 **What to look for:** White areas are concrete and pavement — they dominate the north. Dark green areas are forests and parks — they cluster in the south. The pattern is the **mirror image** of the LST map from Notebook 01.

## ⚖️ A tale of two Mexicos: north vs south

We use the same municipio groups as Notebook 01:

- **Nororiente (hotter in LST):** Gustavo A. Madero, Iztapalapa, Ecatepec de Morelos, Tlalnepantla de Baz, Nezahualcóyotl
- **Sur (cooler in LST):** Coyoacán, Álvaro Obregón, Tlalpan, La Magdalena Contreras

If vegetation is part of the explanation for the temperature gap, we should see **higher mean NDVI in the south** than in the nororiente — in both seasons.

In [ ]:
nororiente_names = [
    "Gustavo A. Madero", "Iztapalapa", "Ecatepec de Morelos",
    "Tlalnepantla de Baz", "Nezahualcóyotl",
]
sur_names = [
    "Coyoacán", "Álvaro Obregón", "Tlalpan", "La Magdalena Contreras",
]

nororiente = aoi.filter(ee.Filter.inList("NOMGEO", nororiente_names))
sur        = aoi.filter(ee.Filter.inList("NOMGEO", sur_names))

def mean_band(image, fc, band):
    return image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=fc.geometry().dissolve(),
        scale=30, maxPixels=1e9,
    ).getInfo()[band]

m_n_s = mean_band(ndvi_summer, nororiente, "NDVI")
m_s_s = mean_band(ndvi_summer, sur,        "NDVI")
m_n_w = mean_band(ndvi_winter, nororiente, "NDVI")
m_s_w = mean_band(ndvi_winter, sur,        "NDVI")

print(f"Summer  — North: {m_n_s:.3f}  |  South: {m_s_s:.3f}  |  Gap: {m_n_s - m_s_s:+.3f}")
print(f"Winter  — North: {m_n_w:.3f}  |  South: {m_s_w:.3f}  |  Gap: {m_n_w - m_s_w:+.3f}")

> 🔑 **Key finding:** The north has **significantly less vegetation** than the south — in both seasons. Using the reference values above, the nororiente's mean NDVI sits in the *residential with some trees* band, while the south sits firmly in the *urban parks and healthy woodland* band. The gap does not close in December — and that, on its own, is a strong clue.

## 🔗 The LST–NDVI relationship

We have a hypothesis: less vegetation goes with more heat. To test it visually, let's pair LST and NDVI at the **same locations** across the AOI and plot them against each other.

The technique: stack the summer LST band and the summer NDVI band into a single image, then sample many random points. Each point gives us one (NDVI, LST) pair. The server does the heavy lifting; the plot itself is plain matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import load_lst_composite

# Reload the summer LST composite so we can pair it with the summer NDVI.
lst_summer = load_lst_composite(
    aoi, start_date="2025-06-01", end_date="2025-09-01", cloud_max=20.0,
)

# Stack the two bands, drop pixels where either is masked.
stacked = ndvi_summer.addBands(lst_summer.rename("LST_C"))

# Sample 400 random points across the AOI (server-side).
samples = stacked.sample(
    region=aoi.geometry(),
    scale=120,           # ~120 m per sample: coarse enough to be fast, fine enough to span the city
    numPixels=400,
    seed=42,
    geometries=False,
).getInfo()

df = pd.DataFrame([f["properties"] for f in samples["features"]]).dropna()
print(f"Sampled points: {len(df)}")

# Plot.
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(
    df["NDVI"], df["LST_C"],
    s=22, alpha=0.45, color="#1a9850",
    edgecolor="white", linewidth=0.4,
)

# Linear fit.
coef = np.polyfit(df["NDVI"], df["LST_C"], 1)
xfit = np.linspace(df["NDVI"].min(), df["NDVI"].max(), 100)
ax.plot(
    xfit, np.polyval(coef, xfit),
    color="#f31a1c", linewidth=2.2,
    label=f"Linear fit: LST = {coef[1]:+.1f} °C  {coef[0]:+.1f}·NDVI",
)

ax.set_xlabel("NDVI (greenness, 0 = none · 1 = dense forest)")
ax.set_ylabel("Summer Land Surface Temperature (°C)")
ax.set_title("Less green, more heat: LST vs NDVI across the ZMVM (summer 2025)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", framealpha=0.95)
plt.tight_layout()
plt.show()

# Strength of the relationship.
r = df["NDVI"].corr(df["LST_C"])
print(f"Pearson correlation: r = {r:.3f}")

> 📊 *Each point is a randomly sampled location in the ZMVM. As vegetation increases (right), surface temperature drops (down). The relationship is clearly negative — and the slope is steep.*

> 🔑 **Key finding (so far):** The north has significantly less vegetation than the south — and where NDVI is low, LST is high. Vegetation explains **part** of the temperature gap, but **likely not all of it**. Other factors — industrial heat, building density, altitude — may also play a role. We'll come back to that question in the synthesis.

## 🧍 A human note

Low NDVI doesn't just mean "fewer parks." It means kids walking to school on **shadeless concrete sidewalks**, bus stops that feel like ovens at 3 pm, and older adults who can't leave their homes in the afternoon because the indoor heat is unbearable. Vegetation isn't decoration — it's infrastructure. And like all infrastructure, it is distributed unevenly across the city.

---

## 🔍 What we know, and what we don't

✅ **Confirmed in this notebook:**
- The northern ZMVM has **less vegetation** than the south — in both summer and winter, with the gap widening in the dry months.
- **NDVI and LST are strongly, negatively correlated** across the metro area: less green = more heat.

❓ **Still open questions:**

- Does the lack of trees **fully** explain the 5–8 °C gap, or are other factors (industry, building density, altitude) also at play?
- Is the less-green north also the more polluted north? → **Notebook 3: NO₂**
- Do poorer neighborhoods systematically have fewer trees? → **Notebook 5: Marginalization**

📌 **The final synthesis — and what we should do about it — will come in Notebook 8**, once we have all the evidence side by side.

---

### 📚 Learn more
- NASA Landsat 8 / 9 — https://landsat.gsfc.nasa.gov/landsat-8-9/
- USGS — *Landsat Collection 2 Level 2 Science Products*
- INEGI — *Censo de Población y Vivienda 2020*
- WHO — *Urban green spaces and health* (2016)

### 🛠️ About the code
- All reusable helpers live in the `src/` package (`aoi.py`, `landsat.py`, `visualization.py`, `config.py`).
- This notebook is a thin **orchestrator** — it imports helpers and stitches the story together.
- Reproducibility: pinned versions in `requirements.txt`; the `src/` package is installed in editable mode.

---

➡️ **Next notebook:** *The Air You Breathe — Mapping NO₂ Pollution Across CDMX*